In [1]:

import numpy as np
import pandas as pd

from pysr import PySRRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score



Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [2]:

# ============================================================
# LOAD DATA
# ============================================================

data = np.load(
    "../../data/c_a/processed/dataset_full.npz",
    allow_pickle=True
)

X_full = data["X"]

df = pd.DataFrame(
    X_full,
    columns=[f"F{i}" for i in range(30)]
)

print("Original shape:", df.shape)

# ============================================================
# OPTIONAL SUBSAMPLING
# ============================================================


Original shape: (6205696, 30)


In [3]:

MAX_SAMPLES = 2500

if len(df) > MAX_SAMPLES:
    df = df.sample(
        n=MAX_SAMPLES,
        random_state=42
    )

print("Using shape:", df.shape)

Using shape: (2500, 30)


In [4]:


# ============================================================
# SCALE DATA
# ============================================================

scaler = StandardScaler()

# X_scaled = scaler.fit_transform(df)
X_scaled = df.values.astype(np.float64)


In [5]:

# ============================================================
# CHOOSE TARGET FEATURE
# ============================================================

# Example:
# Try discovering F3 from all other features

TARGET_FEATURE = 3

y = X_scaled[:, TARGET_FEATURE]

X_other = np.delete(
    X_scaled,
    TARGET_FEATURE,
    axis=1
)

In [6]:


# ============================================================
# FEATURE MAPPING
# ============================================================

remaining_features = [
    i for i in range(X_scaled.shape[1])
    if i != TARGET_FEATURE
]

print("\nFeature mapping:")

for local_idx, original_idx in enumerate(remaining_features):
    print(f"x{local_idx} -> F{original_idx}")



Feature mapping:
x0 -> F0
x1 -> F1
x2 -> F2
x3 -> F4
x4 -> F5
x5 -> F6
x6 -> F7
x7 -> F8
x8 -> F9
x9 -> F10
x10 -> F11
x11 -> F12
x12 -> F13
x13 -> F14
x14 -> F15
x15 -> F16
x16 -> F17
x17 -> F18
x18 -> F19
x19 -> F20
x20 -> F21
x21 -> F22
x22 -> F23
x23 -> F24
x24 -> F25
x25 -> F26
x26 -> F27
x27 -> F28
x28 -> F29


In [ ]:

results = []
NONLINEAR_TOKENS = [
    "square",
    "cube",
    "sin",
    "cos",
    "exp",
    "log",
    "sqrt",
    "max",
    "min",
]
# ============================================================
# LOOP OVER ALL FEATURES
# ============================================================

for TARGET_FEATURE in [10]:
# range(X_scaled.shape[1]):

    print("\n" + "=" * 80)
    print(f"TARGET FEATURE: F{TARGET_FEATURE}")
    print("=" * 80)

    # --------------------------------------------------------
    # TARGET
    # --------------------------------------------------------

    y = X_scaled[:, TARGET_FEATURE]

    # --------------------------------------------------------
    # INPUT FEATURES
    # --------------------------------------------------------

    X_other = np.delete(
        X_scaled,
        TARGET_FEATURE,
        axis=1
    )

    # --------------------------------------------------------
    # FEATURE MAPPING
    # --------------------------------------------------------

    remaining_features = [
        i for i in range(X_scaled.shape[1])
        if i != TARGET_FEATURE
    ]

    print("\nFeature mapping:")

    for local_idx, original_idx in enumerate(remaining_features):
        print(f"x{local_idx} -> F{original_idx}")

    # --------------------------------------------------------
    # TRAIN / TEST SPLIT
    # --------------------------------------------------------

    X_train, X_test, y_train, y_test = train_test_split(
        X_other,
        y,
        test_size=0.2,
        random_state=42,
    )

    # --------------------------------------------------------
    # SYMBOLIC REGRESSION
    # --------------------------------------------------------

    model = PySRRegressor(

        # ----------------------------------------------------
        # SEARCH
        # ----------------------------------------------------

        niterations=2000,
        populations=20, 
        population_size=1000,
        # ----------------------------------------------------
        # SIMPLE OPERATORS FIRST
        # ----------------------------------------------------

        binary_operators=[
        "+",
        "-",
        "*",
        "/",
        # "max",
        # "min",
            ],

        # unary_operators=[
        #     "square",
        #     "cube",
        #     "exp",
        #     "log",
        #     "sqrt",
        #     "sin",
        #     "cos",
        # ],
        
        unary_operators = [
            "square",
            "cube",
            "sqrt",
            "cbrt",
            "log",
            "log1p",
            "exp",
            "sin",
            "cos",
            "tanh",
        ],q
        

        # ----------------------------------------------------
        # BATCHING
        # ----------------------------------------------------

        batching=False,
        # batch_size=256,

        # ----------------------------------------------------
        # COMPLEXITY CONTROL
        # ----------------------------------------------------

        maxsize=40,
        maxdepth=20,

        parsimony=1e-6,

        # ----------------------------------------------------
        # MODEL SELECTION
        # ----------------------------------------------------

        model_selection="accuracy",
        weight_randomize=0.05,
        weight_mutate_constant=0.2,
        weight_mutate_operator=0.2,
        weight_swap_operands=0.1,
        # ----------------------------------------------------
        # LOSS
        # ----------------------------------------------------

        elementwise_loss="loss(x, y) = (x - y)^2",

        # ----------------------------------------------------
        # MULTIPROCESSING
        # ----------------------------------------------------

        procs=0,

        # ----------------------------------------------------
        # OUTPUT
        # ----------------------------------------------------

        verbosity=1,
    )

    # --------------------------------------------------------
    # FIT
    # --------------------------------------------------------

    model.fit(X_train, y_train)

    # --------------------------------------------------------
    # PREDICT
    # --------------------------------------------------------

    pred = model.predict(X_test)

    r2 = r2_score(y_test, pred)

    # --------------------------------------------------------
    # ALL EQUATIONS
    # --------------------------------------------------------

    equations_df = model.equations_

    # --------------------------------------------------------
    # LOOP OVER ALL DISCOVERED EQUATIONS
    # --------------------------------------------------------

    for _, row in equations_df.iterrows():

        equation = row["equation"]
        complexity = row["complexity"]
        loss = row["loss"]

        # ----------------------------------------------------
        # NONLINEAR CHECK
        # ----------------------------------------------------

        is_nonlinear = any(
            token in equation
            for token in NONLINEAR_TOKENS
        )

        if not is_nonlinear:
            continue

        # ----------------------------------------------------
        # PREDICT USING THIS SPECIFIC EQUATION
        # ----------------------------------------------------

        pred = model.predict(
            X_test,
            index=row.name
        )

        r2 = r2_score(y_test, pred)

        # ----------------------------------------------------
        # FILTER STRONG EQUATIONS
        # ----------------------------------------------------

        if r2 < 0.99:
            continue

        # ----------------------------------------------------
        # STORE RESULTS
        # ----------------------------------------------------

        results.append({
            "target_feature": TARGET_FEATURE,
            "r2": r2,
            "complexity": complexity,
            "loss": loss,
            "equation": equation,
        })

        # ----------------------------------------------------
        # PRINT
        # ----------------------------------------------------

    # --------------------------------------------------------
    # PRINT RESULTS
    # --------------------------------------------------------

        print("\nRESULTS")
        print("-" * 40)
    
        print("R²:", r2)
        print("Complexity:", complexity)
        print("Loss:", loss)
    
        print("\nEquation:")
        print(f"F{TARGET_FEATURE} =", equation)



TARGET FEATURE: F10

Feature mapping:
x0 -> F0
x1 -> F1
x2 -> F2
x3 -> F3
x4 -> F4
x5 -> F5
x6 -> F6
x7 -> F7
x8 -> F8
x9 -> F9
x10 -> F11
x11 -> F12
x12 -> F13
x13 -> F14
x14 -> F15
x15 -> F16
x16 -> F17
x17 -> F18
x18 -> F19
x19 -> F20
x20 -> F21
x21 -> F22
x22 -> F23
x23 -> F24
x24 -> F25
x25 -> F26
x26 -> F27
x27 -> F28
x28 -> F29


/home/mariuszoslaw/uni/masters/venv/lib/python3.13/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 0.000e+00
Progress: 0 / 40000 total iterations (0.000%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
───────────────────────────────────────────────────────────────────────────────────────────────────
════════════════════════════════════════════════════════════════════════════════════════════════════
Press 'q' and then <enter> to stop execution early.

Expressions evaluated per second: 0.000e+00
Progress: 0 / 40000 total iterations (0.000%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
─────────────────────────────────────────────────────────────────────────────

In [8]:
# ============================================================
# FINAL SUMMARY
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["r2", "complexity"],
    ascending=[False, True]
)

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

print(results_df)

# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(
    "pysr_feature_dependencies.csv",
    index=False
)

print("\nSaved:")
print("pysr_feature_dependencies.csv")



FINAL SUMMARY
   target_feature        r2  complexity       loss  \
8              10  0.999970          19  23.037888   
5              10  0.999970          16  23.075083   
9              10  0.999970          20  23.032970   
6              10  0.999970          17  23.065039   
7              10  0.999970          18  23.063454   
4              10  0.999970          14  23.096441   
3              10  0.999970          13  23.232319   
2              10  0.999970          12  23.338202   
1              10  0.999969           8  23.652576   
0              10  0.999969           6  23.832355   

                                            equation  
8  (((x26 * 0.32788178) - (x0 / -39.97924)) + -42...  
5  (((x6 + 0.6997032) * 0.32788178) - ((x0 / -39....  
9  (((x26 * 0.32788178) - (x0 / -39.97924)) + -42...  
6  (((x6 + 0.6997032) * 0.32788178) - ((x0 / -39....  
7  (((x6 + 0.6997032) * 0.32788178) - ((x0 / -39....  
4  ((x6 * 0.32788813) + -421.83325) - ((x0 / -39....  
3  ((